In [23]:
import pandas as pd
import numpy as np
import pathlib
from sklearn.model_selection import train_test_split

In [58]:
data_path = pathlib.Path("../data/octmnist_64.npz") #Re specify location if dataset in a different Location

In [59]:
data = np.load(data_path)
print(data.files)
print(data['train_images'].shape)
print(data['val_images'].shape)
print(data['test_images'].shape)


['train_images', 'train_labels', 'val_images', 'val_labels', 'test_images', 'test_labels']
(97477, 64, 64)
(10832, 64, 64)
(1000, 64, 64)


In [21]:
#Concatenating the train, validation and test images into a single array
X = np.concatenate([data["train_images"],data["val_images"],data["test_images"]], axis=0)
y = np.concatenate([data["train_labels"],data["val_labels"],data["test_labels"]], axis=0)
print(X.shape)
y = y.ravel() #Flattening the labels array to make it 1D
print(y.shape)

label_map = {0: "CNV",1: "DME",2: "Drusen",3: "Normal"}
class_counts = pd.Series(y).value_counts().sort_index()
class_distribution = pd.DataFrame({
    "Label": class_counts.index,
    "Class": [label_map[i] for i in class_counts.index],
    "Count": class_counts.values,
    "Percentage": class_counts.values / len(y) * 100
})
class_distribution

(109309, 64, 64)
(109309,)


,Label,Class,Count,Percentage
0,0,CNV,37455,34
1,1,DME,11598,10
2,2,Drusen,8866,8
3,3,Normal,51390,47


In [24]:
X_train, X_temp, y_train, y_temp = train_test_split(X,y,test_size=0.30,random_state=42,stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp,y_temp,test_size=0.50,random_state=42,stratify=y_temp)

In [26]:
total = len(X)
print("Train %:", len(X_train) / total * 100)
print("Validation %:", len(X_val) / total * 100)
print("Test %:", len(X_test) / total * 100)

Train %: 69.99972554867394
Validation %: 14.999679806786267
Test %: 15.000594644539792


In [27]:
def get_distribution(labels, split_name):
    counts = pd.Series(labels).value_counts().sort_index()

    return pd.DataFrame({
        "Split": split_name,
        "Label": counts.index,
        "Class": [label_map[i] for i in counts.index],
        "Count": counts.values,
        "Percentage": counts.values / len(labels) * 100
    })

In [30]:
distribution_df = pd.concat([
    get_distribution(y_train, "Train"),
    get_distribution(y_val, "Validation"),
    get_distribution(y_test, "Test")
])

distribution_df

,Split,Label,Class,Count,Percentage
0,Train,0,CNV,26218,34
1,Train,1,DME,8119,10
2,Train,2,Drusen,6206,8
3,Train,3,Normal,35973,47
0,Validation,0,CNV,5618,34
1,Validation,1,DME,1740,10
2,Validation,2,Drusen,1330,8
3,Validation,3,Normal,7708,47
0,Test,0,CNV,5619,34
1,Test,1,DME,1739,10


In [31]:
print("Data type:", X_train.dtype)
print("Minimum pixel value:", X_train.min())
print("Maximum pixel value:", X_train.max())

print("Train shape:", X_train.shape)
print("Validation shape:", X_val.shape)
print("Test shape:", X_test.shape)

Data type: uint8
Minimum pixel value: 0
Maximum pixel value: 255
Train shape: (76516, 64, 64)
Validation shape: (16396, 64, 64)
Test shape: (16397, 64, 64)


In [32]:
#Adding New axis as Tf expects 4D input for Conv2D layers
X_train = X_train[..., np.newaxis]
X_val = X_val[..., np.newaxis]
X_test = X_test[..., np.newaxis]

In [33]:
print("Train shape:", X_train.shape)
print("Validation shape:", X_val.shape)
print("Test shape:", X_test.shape)

Train shape: (76516, 64, 64, 1)
Validation shape: (16396, 64, 64, 1)
Test shape: (16397, 64, 64, 1)
